In [10]:
import cv2 as cv
import numpy as np

(Podla mna tomu morfologia iba skodi na to sa musime kuknut este ale
pri 3x3 matici to vezme kus intenzity peaku... a 2x2 je useless imo)

In [20]:
# Load image
img = cv.imread("test.png", cv.IMREAD_COLOR)
assert img is not None

# ==========================================================================================
# Preprocessing: PNG → BGR → grayscale → gaussian blur → threshold → processing
# ==========================================================================================

# Apply grayscale to image
gray_image = cv.cvtColor(img, cv.COLOR_BGR2GRAY)

# Apply gaussian blur to image
gauss_image = cv.GaussianBlur(gray_image, (3, 3), 0)

# Apply threshold to image
_, thresh_image = cv.threshold(gauss_image, 0, 255, cv.THRESH_BINARY_INV + cv.THRESH_OTSU)

# Show images
cv.imshow("With grayscale", gray_image)
cv.imshow("With gaussian blur", gauss_image)
cv.imshow("With threshold", thresh_image)
cv.waitKey(0)
cv.destroyAllWindows()



In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# -------------------------------
# CONFIGURATION
# -------------------------------
DATASET_DIR = "../nmr_dataset"          # output directory of build_dataset
INDEX_CSV = os.path.join(DATASET_DIR, "index.csv")
VECTORS_DIR = os.path.join(DATASET_DIR, "vectors")
LABEL_MAP_JSON = os.path.join(DATASET_DIR, "label_map.json")

TEST_SIZE = 0.2                         # fraction of data for testing
RANDOM_STATE = 42
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001
# Model type: "cnn" or "mlp"
MODEL_TYPE = "cnn"
# Input shape (the vector length from generation)
INPUT_SHAPE = (1024, 1)   # 1024 points, 1 channel

# -------------------------------
# LOAD DATA
# -------------------------------
def load_vectors_from_index(index_df, vectors_dir):
    """Load all .npy vector files listed in index_df."""
    X = []
    y = []
    for _, row in index_df.iterrows():
        npy_path = row["npy_path"]
        if not npy_path:
            # Fallback: construct full path
            npy_path = os.path.join(vectors_dir, row["compound"], f"{row['sample_id']}.npy")
        vec = np.load(npy_path).astype(np.float32)
        X.append(vec)
        y.append(row["label"])
    return np.array(X), np.array(y)

print("Loading dataset...")
index_df = pd.read_csv(INDEX_CSV)

# Check if npy_path column exists; if not, we'll construct it
if "npy_path" not in index_df.columns:
    # Assume the paths follow the pattern: vectors/compound_name/sample_id.npy
    index_df["npy_path"] = index_df.apply(
        lambda row: os.path.join(VECTORS_DIR, row["compound"], f"{row['sample_id']}.npy"),
        axis=1
    )

X, y = load_vectors_from_index(index_df, VECTORS_DIR)
print(f"Loaded {len(X)} vectors, each of length {X.shape[1]}")

# Split into train/test (stratified to preserve class distribution)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

# Reshape for CNN: add channel dimension (1D convolution expects shape (batch, steps, channels))
if MODEL_TYPE == "cnn":
    X_train = X_train[..., np.newaxis]   # (n, 1024, 1)
    X_test = X_test[..., np.newaxis]
else:
    # MLP: keep as 2D (n, 1024)
    pass

# Get number of classes
num_classes = len(np.unique(y))
print(f"Number of classes: {num_classes}")

# Convert labels to categorical (one-hot) for neural network
y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes)
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes)

# -------------------------------
# BUILD MODEL
# -------------------------------
if MODEL_TYPE == "cnn":
    model = models.Sequential([
        layers.Conv1D(filters=64, kernel_size=5, activation='relu', input_shape=INPUT_SHAPE),
        layers.MaxPooling1D(pool_size=2),
        layers.Conv1D(filters=128, kernel_size=3, activation='relu'),
        layers.MaxPooling1D(pool_size=2),
        layers.Conv1D(filters=256, kernel_size=3, activation='relu'),
        layers.GlobalAveragePooling1D(),          # reduces to 256 features
        layers.Dropout(0.5),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
else:  # MLP
    model = models.Sequential([
        layers.Flatten(input_shape=(INPUT_SHAPE[0],)),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# -------------------------------
# TRAIN
# -------------------------------
early_stop = callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6
)

history = model.fit(
    X_train, y_train_cat,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.2,   # use 20% of train as validation
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# -------------------------------
# EVALUATE
# -------------------------------
test_loss, test_acc = model.evaluate(X_test, y_test_cat, verbose=0)
print(f"\nTest accuracy: {test_acc:.4f}")

# Plot training history
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train acc')
plt.plot(history.history['val_accuracy'], label='Val acc')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train loss')
plt.plot(history.history['val_loss'], label='Val loss')
plt.title('Loss')
plt.legend()
plt.tight_layout()
plt.savefig("training_history.png")
plt.show()

# -------------------------------
# SAVE MODEL
# -------------------------------
model.save("nmr_classifier.h5")
print("Model saved as 'nmr_classifier.h5'")

# Optionally, save the label map for later inference
with open(LABEL_MAP_JSON, 'r') as f:
    label_map = json.load(f)
# Invert map: index -> compound name
inv_label_map = {v: k for k, v in label_map.items()}
print("Label map:", inv_label_map)